# Lab 12

STAT 109: Demo lab for Lectures 35 and 36 (linear + logistic regression)

# Lab 12: Regression demo (bio/nature examples)

> **Open in Google Colab (R)**
>
> [Open in
> Colab](https://colab.research.google.com/github/roverhol/stat109-public/blob/main/labs/lab-12.ipynb)
> — after it opens, use Runtime -\> Change runtime type and select R if
> needed.

------------------------------------------------------------------------

## Learning outcomes

By the end of this demo lab, you should be able to:

1.  Describe linear association with scatterplots (form, direction,
    strength) and `cor()`.
2.  Explain how correlation changes for weak/moderate/strong positive
    and negative relationships.
3.  Identify when `r` near 0 reflects weak/no **linear** association.
4.  Recognize nonlinear patterns where linear methods should not be
    used.
5.  Fit simple linear regression with `lm()`, make predictions, and
    compute confidence/prediction intervals with `predict()`.
6.  Fit simple logistic regression with `glm(..., family = binomial)`,
    predict probabilities, and use a cutoff for classification.

------------------------------------------------------------------------

## Setup

In [ ]:
if (!requireNamespace("tidyverse", quietly = TRUE)) {
  install.packages("tidyverse", repos = "https://cloud.r-project.org")
}
library(tidyverse)

------------------------------------------------------------------------

## Part 1: Correlation demo across patterns

### A) Simulate several relationship types

In [ ]:
set.seed(1212)
n <- 120
x <- runif(n, 0, 10)

d_pos_strong <- tibble(example = "positive_strong", x = x, y = 2 + 1.9 * x + rnorm(n, 0, 1.0))
d_pos_mod    <- tibble(example = "positive_moderate", x = x, y = 2 + 1.8 * x + rnorm(n, 0, 3.0))
d_pos_weak   <- tibble(example = "positive_weak", x = x, y = 2 + 1.5 * x + rnorm(n, 0, 6.5))

d_neg_strong <- tibble(example = "negative_strong", x = x, y = 24 - 1.8 * x + rnorm(n, 0, 1.0))
d_neg_mod    <- tibble(example = "negative_moderate", x = x, y = 24 - 1.6 * x + rnorm(n, 0, 3.0))
d_neg_weak   <- tibble(example = "negative_weak", x = x, y = 24 - 1.3 * x + rnorm(n, 0, 6.5))

d_none       <- tibble(example = "no_linear_relationship", x = x, y = rnorm(n, 12, 4))
d_nonlinear  <- tibble(example = "nonlinear_curve", x = x, y = 0.7 * (x - 5)^2 + rnorm(n, 0, 1.2))

demo_cor <- bind_rows(
  d_pos_strong, d_pos_mod, d_pos_weak,
  d_neg_strong, d_neg_mod, d_neg_weak,
  d_none, d_nonlinear
)

### B) Correlation table

In [ ]:
cor_table <- demo_cor |>
  group_by(example) |>
  summarise(r = cor(x, y), .groups = "drop") |>
  arrange(desc(r))

cor_table

### C) Scatterplots

In [ ]:
ggplot(demo_cor, aes(x = x, y = y)) +
  geom_point(alpha = 0.75, color = "darkgreen") +
  facet_wrap(~ example, scales = "free_y") +
  labs(
    title = "Relationship type demo (bio/nature themed synthetic data)",
    x = "Explanatory variable (x)",
    y = "Response variable (y)"
  ) +
  theme_minimal(base_size = 11)

Interpretation reminders:

- Closer to `1` or `-1` means stronger **linear** relationship.
- Closer to `0` means weak to no **linear** relationship.
- Sign gives direction (`+` increasing, `-` decreasing).
- For `nonlinear_curve`, do **not** apply linear-correlation/regression
  conclusions as if form were linear.

------------------------------------------------------------------------

## Part 2: Simple linear regression demo (nature example)

Example: predict plant height from sunlight hours.

In [ ]:
set.seed(1213)
plant <- tibble(
  sunlight_hours = runif(140, 2, 12),
  height_cm = 9 + 2.7 * sunlight_hours + rnorm(140, 0, 4)
)

idx <- sample(seq_len(nrow(plant)), size = 0.8 * nrow(plant))
train <- plant[idx, ]
test  <- plant[-idx, ]

### Fit model and interpret

In [ ]:
fit_lin <- lm(height_cm ~ sunlight_hours, data = train)
summary(fit_lin)
coef(fit_lin)

### Predict mean and individual outcomes

In [ ]:
new_x <- tibble(sunlight_hours = 8)

predict(fit_lin, newdata = new_x)  # point prediction
predict(fit_lin, newdata = new_x, interval = "confidence")  # mean response CI
predict(fit_lin, newdata = new_x, interval = "prediction")  # individual PI

### Residual check + test performance

In [ ]:
train <- train |>
  mutate(pred = predict(fit_lin, newdata = train),
         resid = height_cm - pred)

test <- test |>
  mutate(pred = predict(fit_lin, newdata = test))

mse <- function(y, yhat) mean((y - yhat)^2)

tibble(
  train_MSE = mse(train$height_cm, train$pred),
  test_MSE = mse(test$height_cm, test$pred),
  test_R2 = 1 - sum((test$height_cm - test$pred)^2) /
    sum((test$height_cm - mean(test$height_cm))^2)
)

------------------------------------------------------------------------

## Part 3: Simple logistic regression demo (biology example)

Example: model probability that a seed germinates from soil moisture.

In [ ]:
set.seed(1214)
seed <- tibble(
  moisture = runif(180, 5, 45)
) |>
  mutate(
    p_germ = plogis(-3.2 + 0.14 * moisture),
    germinated = rbinom(n(), 1, p_germ)
  )

idx <- sample(seq_len(nrow(seed)), size = 0.8 * nrow(seed))
seed_train <- seed[idx, ]
seed_test  <- seed[-idx, ]

### Fit logistic model + slope CI

In [ ]:
fit_log <- glm(germinated ~ moisture, data = seed_train, family = binomial)
summary(fit_log)
confint(fit_log, "moisture")

### Predicted probabilities and 0.5 cutoff accuracy

In [ ]:
seed_test <- seed_test |>
  mutate(
    p_hat = predict(fit_log, newdata = seed_test, type = "response"),
    y_hat = if_else(p_hat >= 0.5, 1, 0)
  )

tibble(
  test_accuracy = mean(seed_test$germinated == seed_test$y_hat),
  p_at_20 = predict(fit_log, newdata = tibble(moisture = 20), type = "response"),
  p_at_35 = predict(fit_log, newdata = tibble(moisture = 35), type = "response")
)

------------------------------------------------------------------------

## Wrap-up

- Use `cor()` and scatterplots to summarize **linear** association only.
- Use `lm()` + `predict()` for linear prediction and intervals.
- Use `glm(..., family = binomial)` for binary outcomes and probability
  prediction.
- Always check whether linear form is reasonable before applying linear
  methods.